### Dataset setup

The below cells manipulate this data to create a single dataframe which contains information about the county, including health and demographic information, and the percentage of residents that voted for each presidential candidate.

This data is pulled from:
- [County Health Rankings & Roadmaps](https://www.countyhealthrankings.org/)
- [The MIT Election Data and Science Lab](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/VOQCHQ)

In [ ]:
import pandas as pd

In [ ]:
data_location = 's3://mlspace-data-521454461163/project/07NNClassification/datasets/politicalPredictions/'

In [ ]:
df = pd.read_csv(data_location+'analytic_data2023.csv')
cols = list(df.columns[:5])
for col in df.columns:
    if 'raw value' in col or 'Ratio' in col:
        cols.append(col)
cutdown = df[df['County FIPS Code'] != 0][cols]

In [ ]:
elect = pd.read_csv(data_location+'countypres_2000-2020.csv')
elect = elect[elect['year'] == 2020]
elect = elect[['state', 'county_name', 'county_fips', 'candidate', 'party', 'candidatevotes', 'totalvotes']]
elect['vote_ratio']=elect['candidatevotes']/elect['totalvotes']
elect_pivot=elect.pivot_table(index='county_fips',columns='candidate',values='vote_ratio')
elect_pivot=elect_pivot.fillna(0)

In [ ]:
merged=cutdown.merge(elect_pivot,left_on='5-digit FIPS Code',right_index=True,how='outer')
merged.drop('Living Wage raw value',axis=1,inplace=True)
merged.drop('Children Eligible for Free or Reduced Price Lunch raw value',axis=1,inplace=True)
merged.drop('Residential Segregation - Black/White raw value',axis=1,inplace=True)
merged.drop('Child Care Cost Burden raw value',axis=1,inplace=True)
merged.drop('Child Care Centers raw value',axis=1,inplace=True)

In [ ]:
print(merged.columns)
merged.head()

### Assignment

Your job is to use a neural net to build the best predictor of the 2020 election you can.  You should follow the following steps:

- Identify NaNs, and decide what to do with them,
- Split the data into training and testing sets,
- Split off the targets from the training data,
- Construct a neural net with four logit outputs and cross entropy loss to predict the percentage vote for each candidate,
- Use testing error to find the best version of your network.
- Choose some specific counties of interest to you, which are different types of counties (rural/urban, red/blue, minority-majority, etc.).  Find their predictions, and see how accurate those specific counties are.

This will get you up to a 90%.  For the final 10%, perform an *ablation test*, in which you remove inputs, and observe if and by how much worse your predictor becomes.  A steep drop in accuracy upon the removal of an input would suggest it is very important in making this prediction.

Keep in mind **correlation** vs **causation**.  We are discovering correlation, not causation.